Load PDFs ➜ split into chunks (RecursiveCharacterTextSplitter) ➜ embed (HuggingFaceEmbeddings) ➜ store in Chroma

In [2]:
# Load PDFs
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("PDFs/")
docs = loader.load()  # list[Document(page_content, meta_data)], one per page

print(f"{len(docs)} documents loaded")

/var/folders/k2/swnr3fc52yjc83gxdxqzrt6c0000gn/T/ipykernel_6353/1961069663.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


114 documents loaded


In [3]:
# split into chunks (RecursiveCharacterTextSplitter)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)  # list[Document(page_content, meta_data)], metadata kept from each page

print(f"{len(chunks)} chunks created")

549 chunks created


In [4]:
# embed (HuggingFaceEmbeddings)
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # runs locally, 384-dim vectors

vector = embeddings.embed_query(chunks[0].page_content)  # sanity check on one chunk
print(f"embedding dimension: {len(vector)}")

/Users/mesaid/Desktop/Generative AI/Conversational RAG Q&A with PDF Upload and Chat History/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 13:14:45.273000 6353 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14466.51it/s]


embedding dimension: 384


In [5]:
# store in Chroma
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=".chroma/student-rag",  # saved to disk, reload later with Chroma(persist_directory=..., embedding_function=...)
)

print(f"{vectorstore._collection.count()} vectors stored")

549 vectors stored
